# EarningsLens — Stage 3: Forward-Looking Statement Classification

Stage 3 runs every sentence produced by Stage 2 through `finbert-fls`, a FinBERT-based classifier fine-tuned specifically on financial forward-looking statements. Each sentence receives one of three labels:

- **Specific FLS** — a concrete, quantified forward-looking commitment (e.g. *"We expect full-year revenue of $4.2 billion"*)
- **Non-Specific FLS** — a vague or qualitative forward-looking statement (e.g. *"We remain optimistic about the year ahead"*)
- **Not FLS** — a historical statement, conversational fragment, or non-forward-looking content

All 1,954,667 sentences are classified. `Specific FLS` sentences are the primary output — these are the extracted management commitments written into the `commitments` table in Supabase and consumed by Stage 4 (slot filling). `Non-Specific FLS` sentences are retained for supporting context in the RAG pipeline (Stage 8) but do not enter the commitment tracker.

### Why all sentences rather than executive prepared_remarks only

Executives frequently make quantified forward-looking statements in response to analyst questions during Q&A — committing to capex ranges, margin guidance, and unit targets in direct response to analyst pressure. Restricting classification to prepared_remarks would miss this material. The FLS classifier filters non-forward-looking content regardless of speaker or segment, so running on all sentences adds completeness without changing precision.

## 0. Dependencies

`transformers` provides the `finbert-fls` model and tokeniser via the HuggingFace Hub. `torch` is the underlying deep learning framework — the CUDA device is confirmed before inference begins. `tqdm` provides progress bars across the batch loop. All other dependencies were installed in Stage 1–2.

In [ ]:
# %pip install -q transformers torch tqdm python-dotenv psycopg2-binary

### 0.1 Imports & Device Configuration

The CUDA device is confirmed and its name logged before any model loading occurs. If CUDA is unexpectedly unavailable the cell raises immediately rather than silently falling back to CPU and running for days.

In [ ]:
# Standard library
import os
import logging
from pathlib import Path
import hashlib

# Deep learning
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Data
import pandas as pd
import numpy as np
from tqdm import tqdm

# Database
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

c:\Users\sidsu\Desktop\Sem 4\NLP\NLPvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
)
log = logging.getLogger("earningslens.stage3")

In [ ]:
# Confirm CUDA
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available."
    )

In [ ]:
DEVICE = torch.device("cuda")
log.info("CUDA device : %s", torch.cuda.get_device_name(0))
log.info("VRAM        : %.1f GB", torch.cuda.get_device_properties(0).total_memory / 1e9)
print("✓ Device configured")

2026-05-06 22:35:08,252 [INFO] CUDA device : NVIDIA GeForce RTX 4060 Laptop GPU
2026-05-06 22:35:08,253 [INFO] VRAM        : 8.6 GB


✓ Device configured


## 1. Configuration

`BATCH_SIZE` of 64 is a safe default for most NVIDIA GPUs with 6–8 GB VRAM. Increase to 128 if VRAM allows to roughly halve runtime; decrease to 32 if CUDA out-of-memory errors occur.

`CHECKPOINT_PATH` is a local Parquet file that accumulates all classified sentences. If the run is interrupted, the next run detects the checkpoint and resumes from where it stopped — no sentence is reclassified twice.

`MAX_LENGTH` of 128 tokens covers the vast majority of earnings call sentences. Sentences exceeding 128 tokens are truncated at the right, which preserves the metric and value that typically appear early in the sentence.

In [ ]:
# Model
FLS_MODEL_NAME = "yiyanghkust/finbert-fls"
BATCH_SIZE     = 64      # reduce to 32 if CUDA OOM; increase to 128 if VRAM allows
MAX_LENGTH     = 128     # token limit per sentence - covers >99% of corpus sentences

In [ ]:
# Checkpoint — progress saved here every 100k sentences
CHECKPOINT_PATH = Path("fls_checkpoint_2023_2025.parquet")

# Confidence threshold — Specific FLS sentences below this are excluded from Stage 4
FLS_CONFIDENCE_THRESHOLD = 0.75

In [ ]:
# Supabase connection
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise EnvironmentError("DATABASE_URL not found — check your .env file")

In [ ]:
print("✓ Configuration set")
print(f"  Model      : {FLS_MODEL_NAME}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Max tokens : {MAX_LENGTH}")
print(f"  Checkpoint : {CHECKPOINT_PATH}")

✓ Configuration set
  Model      : yiyanghkust/finbert-fls
  Batch size : 64
  Max tokens : 128
  Checkpoint : fls_checkpoint_2023_2025.parquet


## 2. Load sentences_df

`sentences_df` is loaded from the Stage 1–2 session if still in memory, or from the Parquet cache if the kernel was restarted between sessions. If neither is available, Stage 1–2 must be rerun and `sentences_df.to_parquet('sentences_cache_2023_2025.parquet')` called before proceeding.

In [ ]:
# Load sentences
sentences_df = pd.read_parquet("sentences_cache_2023_2025.parquet")

In [ ]:
# Exclude operator sentences from FLS classification - Operators never make forward-looking commitments
before = len(sentences_df)
sentences_df = sentences_df[sentences_df["speaker_role"] != "operator"].copy()
print(f"Operator sentences excluded : {before - len(sentences_df):,}")
print(f"Sentences remaining         : {len(sentences_df):,}")

Operator sentences excluded : 127,521
Sentences remaining         : 1,954,667


In [ ]:
print(f"Total sentences to classify : {len(sentences_df):,}")
print(f"Columns                     : {list(sentences_df.columns)}")

Total sentences to classify : 1,954,667
Columns                     : ['transcript_id', 'ticker', 'company_name', 'quarter', 'year', 'speaker', 'speaker_role', 'segment_type', 'sentence_index', 'sentence']


## 3. Load finbert-fls

`yiyanghkust/finbert-fls` is a FinBERT model fine-tuned on a corpus of financial forward-looking statements. It outputs three classes — `Specific FLS`, `Non-Specific FLS`, and `Not FLS` — via a softmax head over the BERT `[CLS]` token representation.

The model is loaded in `float16` precision to reduce VRAM consumption by approximately half compared to the default `float32`, with no meaningful loss of classification accuracy for a three-class softmax task. `model.eval()` disables dropout layers for inference. `torch.inference_mode()` is used during the batch loop as it is strictly faster than `torch.no_grad()` for inference-only workloads.

In [ ]:
log.info("Loading %s...", FLS_MODEL_NAME)

tokeniser = AutoTokenizer.from_pretrained(FLS_MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    FLS_MODEL_NAME,
    dtype=torch.float32,
)
model.to(DEVICE)
model.eval()

# Log the label mapping to confirm id2label matches expectations before
# the long inference run begins — label ordering varies across model versions
log.info("Label mapping : %s", model.config.id2label)
print("✓ Model loaded")
print(f"  Labels : {model.config.id2label}")

2026-05-06 22:35:09,905 [INFO] Loading yiyanghkust/finbert-fls...
2026-05-06 22:35:10,503 [INFO] HTTP Request: HEAD https://huggingface.co/yiyanghkust/finbert-fls/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-06 22:35:10,526 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/yiyanghkust/finbert-fls/443586dc31c765c0aaf1c4daaed8cf3643c92fa5/config.json "HTTP/1.1 200 OK"
2026-05-06 22:35:10,785 [INFO] HTTP Request: HEAD https://huggingface.co/yiyanghkust/finbert-fls/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-06 22:35:11,028 [INFO] HTTP Request: HEAD https://huggingface.co/yiyanghkust/finbert-fls/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-06 22:35:11,269 [INFO] HTTP Request: GET https://huggingface.co/api/models/yiyanghkust/finbert-fls/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-06 22:35:11,512 [INFO] HTTP Request: GET https://huggingface.co/ap

✓ Model loaded
  Labels : {0: 'Not FLS', 1: 'Non-specific FLS', 2: 'Specific FLS'}


2026-05-06 22:35:14,578 [INFO] HTTP Request: HEAD https://huggingface.co/yiyanghkust/finbert-fls/resolve/refs%2Fpr%2F3/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-05-06 22:35:14,967 [INFO] HTTP Request: HEAD https://huggingface.co/yiyanghkust/finbert-fls/resolve/refs%2Fpr%2F3/model.safetensors "HTTP/1.1 302 Found"


## 4. Checkpoint Resume Logic

If `fls_checkpoint_2023_2025.parquet` exists from a previous interrupted run, the already-classified sentence indices are loaded and excluded from the current run. The classifier picks up exactly where it stopped without reclassifying any sentence. If no checkpoint exists the full corpus is processed from the beginning.

In [ ]:
# Determine which sentences still need classification
if CHECKPOINT_PATH.exists():
    completed    = pd.read_parquet(CHECKPOINT_PATH)
    done_indices = set(completed["orig_index"].tolist())
    sentences_to_classify = sentences_df[~sentences_df.index.isin(done_indices)].copy()
    log.info(
        "Checkpoint found: %d already classified, %d remaining",
        len(completed), len(sentences_to_classify)
    )
else:
    completed             = pd.DataFrame()
    sentences_to_classify = sentences_df.copy()
    log.info("No checkpoint — classifying all %d sentences", len(sentences_to_classify))

print(f"Sentences to classify this run : {len(sentences_to_classify):,}")

2026-05-06 22:35:14,816 [INFO] Checkpoint found: 1954667 already classified, 0 remaining


Sentences to classify this run : 0


## 5. FLS Classification

The classifier runs in batches over all sentences. Each batch is tokenised, moved to CUDA, and passed through `finbert-fls`. The softmax output gives a probability distribution over the three FLS labels — the argmax is taken as the predicted label and the corresponding probability is stored as the confidence score.

Progress is checkpointed to `fls_checkpoint.parquet` every 100,000 sentences. Each checkpoint write appends to any existing checkpoint so the file always contains the complete classified set up to that point. The `sentence` text is truncated at `MAX_LENGTH` tokens for the model input — the full original sentence is preserved in the output.

In [ ]:
def classify_batch(texts: list[str]) -> tuple[list[str], list[float]]:
    """
    Run one batch of sentences through finbert-fls.

    Tokenises, runs forward pass on CUDA, and returns the predicted label
    and softmax confidence for each sentence in the batch.

    Args:
        texts: List of sentence strings.

    Returns:
        labels      : Label strings per sentence ('Specific FLS', 'Non-Specific FLS', 'Not FLS')
        confidences : Softmax probability of the predicted label per sentence
    """
    encoded = tokeniser(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.inference_mode():
        logits = model(**encoded).logits         # shape: (batch_size, 3)

    # Move off GPU immediately to free VRAM for the next batch
    probs      = torch.softmax(logits, dim=-1).cpu()
    pred_ids   = probs.argmax(dim=-1).tolist()
    pred_probs = probs.max(dim=-1).values.tolist()

    labels = [model.config.id2label[i] for i in pred_ids]
    return labels, pred_probs

In [ ]:
# Main classification loop
CHECKPOINT_EVERY = 100_000   # write progress to disk every N sentences

sentences_list = sentences_to_classify["sentence"].tolist()
indices_list   = sentences_to_classify.index.tolist()

all_labels      = []
all_confidences = []
checkpoint_buffer = []   # accumulates rows between checkpoint writes

log.info(
    "Starting FLS classification — %d sentences, batch size %d",
    len(sentences_list), BATCH_SIZE
)

for batch_start in tqdm(range(0, len(sentences_list), BATCH_SIZE), desc="FLS"):

    batch_texts   = sentences_list[batch_start : batch_start + BATCH_SIZE]
    batch_indices = indices_list[batch_start   : batch_start + BATCH_SIZE]

    labels, confidences = classify_batch(batch_texts)

    all_labels.extend(labels)
    all_confidences.extend(confidences)

    # Buffer results for checkpoint
    for idx, label, conf in zip(batch_indices, labels, confidences):
        checkpoint_buffer.append({
            "orig_index"     : idx,
            "fls_label"      : label,
            "fls_confidence" : conf,
        })

    # Write checkpoint every CHECKPOINT_EVERY sentences
    sentences_processed = batch_start + len(batch_texts)
    if sentences_processed % CHECKPOINT_EVERY < BATCH_SIZE:
        checkpoint_new = pd.DataFrame(checkpoint_buffer)
        if CHECKPOINT_PATH.exists():
            existing       = pd.read_parquet(CHECKPOINT_PATH)
            checkpoint_new = pd.concat([existing, checkpoint_new], ignore_index=True)
        checkpoint_new.to_parquet(CHECKPOINT_PATH, index=False)
        checkpoint_buffer = []   # reset buffer after write
        log.info(
            "Checkpoint saved — %d / %d sentences classified",
            sentences_processed, len(sentences_list)
        )

# Write any remaining buffer after the loop ends
if checkpoint_buffer:
    checkpoint_new = pd.DataFrame(checkpoint_buffer)
    if CHECKPOINT_PATH.exists():
        existing       = pd.read_parquet(CHECKPOINT_PATH)
        checkpoint_new = pd.concat([existing, checkpoint_new], ignore_index=True)
    checkpoint_new.to_parquet(CHECKPOINT_PATH, index=False)
    log.info("Final checkpoint saved")

log.info("Classification complete")

2026-05-06 22:35:14,834 [INFO] Starting FLS classification — 0 sentences, batch size 64
FLS: 0it [00:00, ?it/s]
2026-05-06 22:35:14,871 [INFO] Classification complete


## 6. Attach Labels to sentences_df

The FLS labels and confidence scores are joined back onto `sentences_df` using the original DataFrame index as the join key, producing `classified_df`. Any sentences classified in prior checkpoint runs are merged in at this point so `classified_df` always reflects the complete classification state.

In [ ]:
# Load the complete checkpoint — covers this run and any prior partial runs
checkpoint_df = pd.read_parquet(CHECKPOINT_PATH)

# Merge labels back onto sentences_df via the original index
classified_df = sentences_df.copy()
classified_df["fls_label"]      = None
classified_df["fls_confidence"] = None

classified_df.loc[checkpoint_df["orig_index"].values, "fls_label"] = \
    checkpoint_df["fls_label"].values
classified_df.loc[checkpoint_df["orig_index"].values, "fls_confidence"] = \
    checkpoint_df["fls_confidence"].values

print(f"Classified   : {classified_df['fls_label'].notna().sum():,}")
print(f"Unclassified : {classified_df['fls_label'].isna().sum():,}")

Classified   : 1,954,667
Unclassified : 0


## 7. Classification Results

The distribution of FLS labels across the full corpus is examined. Expected approximate proportions based on the literature and prior `finbert-fls` benchmarks on earnings call corpora:

- `Not FLS` — ~70–75% (historical statements, conversational turns, greetings)
- `Non-Specific FLS` — ~15–20% (qualitative forward-looking language)
- `Specific FLS` — ~5–10% (quantified commitments — the pipeline target)

The `Specific FLS` count determines the volume of commitment candidates entering Stage 4 slot filling. The project objective targets greater than 85% precision in detecting specific management commitments.

In [ ]:
print("=" * 54)
print("  EarningsLens — Stage 3 Classification Results")
print("=" * 54)

label_counts = classified_df["fls_label"].value_counts()
total        = len(classified_df)

for label, count in label_counts.items():
    pct = count / total * 100
    print(f"  {label:<22}: {count:>9,}  ({pct:.1f}%)")

specific_fls = classified_df[classified_df["fls_label"] == "Specific FLS"]

print(f"\nSpecific FLS sentences       : {len(specific_fls):,}")
print(f"  → These enter Stage 4 slot filling")
print(f"\nMean confidence (all)        : {classified_df['fls_confidence'].mean():.3f}")
print(f"Mean confidence (Specific)   : {specific_fls['fls_confidence'].mean():.3f}")

print("\n── Specific FLS by speaker role ──")
print(specific_fls["speaker_role"].value_counts().to_string())

print("\n── Specific FLS by segment type ──")
print(specific_fls["segment_type"].value_counts().to_string())

print("\n── Sample Specific FLS sentences ──")
sample = specific_fls["sentence"].sample(8, random_state=42)
for i, s in enumerate(sample, 1):
    print(f"  {i}. {s[:140]}")

  EarningsLens — Stage 3 Classification Results
  Not FLS               : 1,671,140  (85.5%)
  Specific FLS          :   195,578  (10.0%)
  Non-specific FLS      :    87,949  (4.5%)

Specific FLS sentences       : 195,578
  → These enter Stage 4 slot filling

Mean confidence (all)        : 0.883
Mean confidence (Specific)   : 0.716

── Specific FLS by speaker role ──
speaker_role
executive    108620
analyst       86958

── Specific FLS by segment type ──
segment_type
prepared_remarks    108616
qa                   86962

── Sample Specific FLS sentences ──
  1. And so, I think you're going to see growth for the next several years.
  2. Based on our current planning assumptions, we anticipate full year adjusted operating profit margins to be in the top half of our target ran
  3. We also expect distributor restock orders will pick up as they receive the last constrained items that allow them to clear their committed i
  4. While we had COVID-19 vaccine sales in the second quarter, we do

## 7.1 Confidence Threshold Filter

The raw `Specific FLS` label set includes borderline cases that sit close to the Non-Specific/Specific decision boundary. Sentences like historical statements framed as forward-looking, qualitative confidence expressions, and accounting disclosures receive `Specific FLS` labels with low softmax confidence (0.50–0.74).

A confidence threshold of **0.75** removes these borderline cases before Stage 4 slot filling. This improves slot filling quality — the LLM is presented with cleaner, higher-signal sentences — at the cost of a modest reduction in recall. Sentences below the threshold are retained in `classified_df` and are available for the RAG pipeline (Stage 8b) but do not enter the commitment tracker.

`FLS_CONFIDENCE_THRESHOLD` is defined in the Configuration cell (Section 1) and can be adjusted there if a different precision/recall trade-off is preferred.

In [ ]:
# Filter to high-confidence Specific FLS only
specific_fls_filtered = classified_df[
    (classified_df["fls_label"] == "Specific FLS") &
    (classified_df["fls_confidence"] >= FLS_CONFIDENCE_THRESHOLD)
]

print(f"Before threshold filter : {len(specific_fls):,}")
print(f"After threshold filter  : {len(specific_fls_filtered):,}")
dropped = len(specific_fls) - len(specific_fls_filtered)
print(f"Dropped (borderline)    : {dropped:,}  ({dropped/len(specific_fls)*100:.1f}%)")
print(f"\nMean confidence (before) : {specific_fls['fls_confidence'].mean():.3f}")
print(f"Mean confidence (after)  : {specific_fls_filtered['fls_confidence'].mean():.3f}")
print(f"\n── Filtered Specific FLS by speaker role ──")
print(specific_fls_filtered["speaker_role"].value_counts().to_string())
print(f"\n── Filtered Specific FLS by segment type ──")
print(specific_fls_filtered["segment_type"].value_counts().to_string())
print(f"\n── Sample filtered Specific FLS sentences ──")
sample = specific_fls_filtered["sentence"].sample(min(6, len(specific_fls_filtered)), random_state=42)
for i, s in enumerate(sample, 1):
    print(f"  {i}. {s[:140]}")

Before threshold filter : 195,578
After threshold filter  : 92,530
Dropped (borderline)    : 103,048  (52.7%)

Mean confidence (before) : 0.716
Mean confidence (after)  : 0.867

── Filtered Specific FLS by speaker role ──
speaker_role
executive    70521
analyst      22009

── Filtered Specific FLS by segment type ──
segment_type
prepared_remarks    70594
qa                  21936

── Sample filtered Specific FLS sentences ──
  1. We remain confident in the advanced ratemaking process and look forward to submitting additional filings following our stakeholder and regul
  2. So overall, 2025 should be another year of significant growth with more than 90% of our end markets seeing positive growth.
  3. Approximately 48% of the 600 and $2.5 million in sales proceeds expected from these transactions come from our land bank and are expected to
  4. Approximately 49% of total RPO is expected to be recognized as revenue over the next 12 months.
  5. All up, our AI business is on track to surpa

In [ ]:
# ── Dedup duplicate (transcript_id, sentence) pairs ──────────────────────────
before = len(specific_fls_filtered)
specific_fls_filtered = specific_fls_filtered.drop_duplicates(
    subset=["transcript_id", "sentence"],
    keep="first"
).copy()
dupes_removed = before - len(specific_fls_filtered)
log.info("Deduped %d duplicate (transcript_id, sentence) rows", dupes_removed)
print(f"\nDuplicates removed      : {dupes_removed}")
print(f"Final insert count      : {len(specific_fls_filtered):,}")

2026-05-06 22:35:15,988 [INFO] Deduped 72 duplicate (transcript_id, sentence) rows



Duplicates removed      : 72
Final insert count      : 92,458


## 8. Write Specific FLS to Supabase

`Specific FLS` sentences are written into the `commitments` table in Supabase. Each row contains the source sentence and its transcript metadata. The `metric`, `value`, `timeframe`, and `hedge_score` columns remain NULL at this stage — these are populated by Stage 4 (slot filling) and Stage 5 (hedge scoring) respectively.

A unique constraint on `(transcript_id, sentence_text)` is added before the insert to prevent duplicate commitment rows from accumulating across reruns. `ON CONFLICT DO NOTHING` makes the insert safe to rerun.

In [ ]:
specific_fls_filtered.to_parquet("specific_fls_cache.parquet", index=False)
print(f"Saved {len(specific_fls_filtered):,} rows to specific_fls_cache.parquet")
print(f"Columns: {list(specific_fls_filtered.columns)}")

Saved 92,458 rows to specific_fls_cache.parquet
Columns: ['transcript_id', 'ticker', 'company_name', 'quarter', 'year', 'speaker', 'speaker_role', 'segment_type', 'sentence_index', 'sentence', 'fls_label', 'fls_confidence']


In [ ]:
specific_fls_filtered = pd.read_parquet("specific_fls_cache.parquet")
print(f"Reloaded {len(specific_fls_filtered):,} rows from specific_fls_cache.parquet")

Reloaded 92,458 rows from specific_fls_cache.parquet


In [ ]:
conn   = psycopg2.connect(DATABASE_URL + "?sslmode=require")
cursor = conn.cursor()

cursor.execute("""
    DO $$
    BEGIN
        IF NOT EXISTS (
            SELECT 1 FROM pg_constraint
            WHERE conname = 'uq_commitment_sentence'
        ) THEN
            ALTER TABLE commitments
            ADD CONSTRAINT uq_commitment_sentence
            UNIQUE (transcript_id, sentence_text);
        END IF;
    END;
    $$;
""")
conn.commit()

In [ ]:
cursor.execute("SELECT ticker, company_id FROM companies")
ticker_to_id = {row[0]: row[1] for row in cursor.fetchall()}

# Build insert rows — now includes sentence_hash
commit_rows = [
    (
        row["transcript_id"],
        ticker_to_id.get(row["ticker"]),
        int(row["quarter"]),
        int(row["year"]),
        row["sentence"],
        hashlib.md5(f"{row['transcript_id']}||{row['sentence']}".encode("utf-8")).hexdigest(),
        None, None, None, None,
        row["fls_label"],
        "Pending",
        None,
    )
    for _, row in specific_fls_filtered.iterrows()
]

log.info("Inserting %d Specific FLS commitments...", len(commit_rows))

execute_values(
    cursor,
    """
    INSERT INTO commitments
        (transcript_id, company_id, quarter, year, sentence_text, sentence_hash,
         metric, value, timeframe, hedge_score, fls_label, status, matched_commitment_id)
    VALUES %s
    ON CONFLICT (transcript_id, sentence_text) DO NOTHING
    """,
    commit_rows,
    page_size=1000,
)

cursor.execute("SELECT COUNT(*) FROM commitments")
count = cursor.fetchone()[0]
print(f"Commitments table : {count:,} rows")

conn.commit()
cursor.close()
conn.close()
print("✓ Stage 3 complete — Supabase updated")

2026-05-06 22:35:21,282 [INFO] Inserting 92458 Specific FLS commitments...


Commitments table : 92,458 rows
✓ Stage 3 complete — Supabase updated


## Handoff to Stage 4

Stage 3 is complete. The outputs are:

| Output | Location | Consumed by |
|---|---|---|
| `classified_df` | In memory | Stage 4 reference |
| `fls_checkpoint.parquet` | Local disk | Resume on interruption |
| `commitments` (Specific FLS rows) | Supabase | Stage 4 — slot filling |

Stage 4 reads each `Specific FLS` sentence from the `commitments` table and runs a two-pass slot filling extraction — spaCy for structured metric/value/timeframe patterns, followed by an Ollama LLM call for anything spaCy does not resolve — to populate the `metric`, `value`, and `timeframe` columns that remain NULL at this stage.